In [ ]:
## layer 2

In [ ]:
import torch 


# input of layer 2
Leye = torch.rand(128, 8, 8)
print(Leye)
Reye = torch.rand(128, 8, 8)
print(Reye)
FaceData = torch.rand(128, 8, 8)  # currently matched with eyes......
print(FaceData)

class TinyModel(torch.nn.Module):
    def __init__(self):
        super(TinyModel, self).__init__()
        # layer 1: feature extraction (to be implemented)
        
        
        # layer 2: feature fusion: concate + group  normalization
        # self.concate = torch.cat((x, x, x), 0) // not needed here, only in forward
        # total_ch = Leye[1]+Reye[1]+FaceData[1]  # total channels of the input // this is not working 
        total_ch = 384
        self.gn = torch.nn.GroupNorm(3, total_ch)     # Separate 6 channels into 3 groups, how to define a appropriate number of groups & channels?
        
        # layer 3:self-attention
        self.norm = torch.nn.LayerNorm([total_ch, 8, 8])
        self.mha = torch.nn.MultiheadAttention(total_ch, num_heads=1, batch_first=True)
        self.scale = torch.nn.Parameter(torch.zeros(1))

    def use_attention(self, x):
        # Reshape input for multi-head attention
        bs, c, h, w = x.shape
        x_att = x.reshape(bs, c, h * w).transpose(1, 2)  # BSxHWxC
        
        # Apply Layer Normalization
        x_att = self.norm(x_att)
        # Apply Multi-head Attention
        att_out, att_map  = self.mha(x_att, x_att, x_att)  # Q,K,V  Q=K=V, self-attention
        return att_out.transpose(1, 2).reshape(bs, c, h, w), att_map # transpose first, then Reshape output to original shape (bs, c, h, w)


    def forward(self, left_eye, right_eye, face):

        # layer2
        concate = torch.cat((left_eye, right_eye, face), 1)  # dim = 0 or 1?  only channel dim changes?
        out = self.gn(concate)
        att_out, _ = self.use_attention(out)
        out = out + self.scale * att_out
        return out 
        

model = TinyModel()
print(model)

output = model(Leye, Reye, FaceData)  # currently the face Data is not matched with eyes, should be matched in the future, how to match? linear padding or other methods? would it affect the performance if change the size of facedata
print("output:",output.shape)


tensor([[[[0.2046, 0.3697, 0.8376,  ..., 0.9361, 0.0732, 0.7744],
          [0.3735, 0.6868, 0.3603,  ..., 0.8635, 0.3515, 0.0741],
          [0.2533, 0.1458, 0.1344,  ..., 0.5381, 0.5727, 0.9165],
          ...,
          [0.9439, 0.9793, 0.7057,  ..., 0.8174, 0.8716, 0.7852],
          [0.2984, 0.8004, 0.1952,  ..., 0.6565, 0.0400, 0.6618],
          [0.4790, 0.6896, 0.2336,  ..., 0.0238, 0.0089, 0.6758]],

         [[0.4460, 0.9305, 0.4664,  ..., 0.9804, 0.5461, 0.1431],
          [0.5412, 0.1228, 0.3978,  ..., 0.3507, 0.4361, 0.3656],
          [0.4667, 0.5687, 0.2427,  ..., 0.6028, 0.5481, 0.4875],
          ...,
          [0.8545, 0.6983, 0.4718,  ..., 0.9281, 0.6738, 0.9238],
          [0.3658, 0.5188, 0.6391,  ..., 0.8130, 0.8906, 0.5162],
          [0.9662, 0.9367, 0.8047,  ..., 0.2171, 0.2119, 0.9740]],

         [[0.0249, 0.0658, 0.1007,  ..., 0.7124, 0.9012, 0.9648],
          [0.1093, 0.2427, 0.1579,  ..., 0.9513, 0.4961, 0.8841],
          [0.2451, 0.3460, 0.1035,  ..., 0

RuntimeError: Given normalized_shape=[384, 8, 8], expected input with shape [*, 384, 8, 8], but got input of size[1, 64, 384]